In [17]:
import os
import glob
from bs4 import BeautifulSoup
import pandas as pd
import re

In [18]:
def clean_text(s: str) -> str:
    s = (s or "").replace("\xa0", " ").replace("&nbsp;", " ")
    return re.sub(r"\s+", " ", s).strip()

def meaningful(txt: str) -> bool:
    if not txt:
        return False
    if re.fullmatch(r"[•\-\u2022\.\,]*", txt):
        return False
    return bool(re.search(r"\w", txt))

def heading_text(tag) -> str:
    for bad in tag.select('.copy-anchor, a.anchor, a[id^="anchor-"], span[id^="anchor-"]'):
        bad.decompose()
    txt = clean_text(tag.get_text(" ", strip=True))
    txt = re.sub(r"\s*#\S+\s*$", "", txt)
    return txt

def parse_file(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read())
    file_chapter = path.replace(".html", "").split("/")[-1]

    h1 = soup.find("h1")
    chapter = clean_text(h1.get_text(" ", strip=True)) if h1 else file_chapter

    root = soup.select_one(".box.introbox.body.field-body .body-inner")
    if root is None:
        root = soup.select_one("article") or soup.body

    rows = []
    current_section = None

    for el in root.descendants:
        if not getattr(el, "name", None):
            continue
        tag = el.name.lower()

        if tag in ("h2", "h3", "h4", "h5"):
            sec = heading_text(el)
            if meaningful(sec):
                current_section = sec

        elif tag == "p":
            txt = clean_text(el.get_text(" ", strip=True))
            if meaningful(txt):
                section = f"{chapter} — {current_section}" if current_section else chapter
                rows.append({"section": section, "paragraph": txt})

        elif tag == "li":
            txt = clean_text(el.get_text(" ", strip=True))
            if meaningful(txt):
                section = f"{chapter} — {current_section}" if current_section else chapter
                rows.append({"section": section, "paragraph": f"• {txt}"})

    df = pd.DataFrame(rows, columns=["section", "paragraph"]).drop_duplicates().reset_index(drop=True)
    return df

In [19]:
INPUT_DIR = "book"
OUTPUT_DIR = "data/parsed_paragraphs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

files = sorted(set(
    glob.glob(os.path.join(INPUT_DIR, "*.html"))
))

if not files:
    print("No HTML files found in:", INPUT_DIR)

for path in files:
    base = os.path.splitext(os.path.basename(path))[0]
    out_csv = os.path.join(OUTPUT_DIR, f"{base}.csv")
    try:
        df = parse_file(path)
        df.to_csv(out_csv, index=False, encoding="utf-8")
        print(f"[OK] {base}: {len(df)} rows -> {out_csv}")
    except Exception as e:
        print(f"[ERROR] {base}: {e}")

[OK] Chapter 10_ Analysing data and undertaking meta-analyses _ Cochrane: 352 rows -> data/parsed_paragraphs\Chapter 10_ Analysing data and undertaking meta-analyses _ Cochrane.csv
[OK] Chapter 11_ Undertaking network meta-analyses _ Cochrane: 287 rows -> data/parsed_paragraphs\Chapter 11_ Undertaking network meta-analyses _ Cochrane.csv
[OK] Chapter 12_ Synthesizing and presenting findings using other methods _ Cochrane: 191 rows -> data/parsed_paragraphs\Chapter 12_ Synthesizing and presenting findings using other methods _ Cochrane.csv
[OK] Chapter 13_ Assessing risk of bias due to missing evidence in a meta-analysis _ Cochrane: 191 rows -> data/parsed_paragraphs\Chapter 13_ Assessing risk of bias due to missing evidence in a meta-analysis _ Cochrane.csv
[OK] Chapter 14_ Completing ‘Summary of findings’ tables and grading the certainty of the evidence _ Cochrane: 321 rows -> data/parsed_paragraphs\Chapter 14_ Completing ‘Summary of findings’ tables and grading the certainty of the e